In [19]:
from utils.decoding_init import apply_style, visualize

apply_style()

In [16]:
# --- Hallucinated Response ---
import pickle

with open("hallucination_response.pkl", "rb") as f:
    response = pickle.load(f)

In [17]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path='.env', override=True)
client = OpenAI()
[m.id for m in client.models.list()]

['Qwen3 235, the best model as of August 2025',
 '1 - GPT-OSS-120b - an open model released by OpenAI in August 2025',
 '7 - Qwen3-Coder-30B-A3B-Instruct - A code model from August 2025',
 'Tongyi-DeepResearch-30B-A3B',
 'alias-huge',
 'Qwen3 Coder 30B with function call',
 'Qwen3-Next',
 '15 Apertus-8B-Instruct-2509 - A new swiss model from September 2025',
 'alias-apertus',
 'alias-large',
 'alias-code',
 'alias-function-call',
 'Qwen3-VL-32B-Instruct-FP8',
 'alias-fast',
 'Phi-4-multimodal-instruct',
 '1 - Ministral 8b - the fast model']

In [13]:
model = "1 - Ministral 8b - the fast model"

# Decoding

When a language model generates text, it doesn’t *choose words directly*.
Instead, it predicts a **distribution over the vocabulary** for the next token.

### How it works
1. The model produces **one logit per token** in the vocabulary (e.g., 50,000 possible tokens).  
2. These logits are passed through a **softmax** function:
$$  P_i = \frac{e^{z_i / \tau}}{\sum_j e^{z_j / \tau}} $$
where $z_i$ is the logit for token *i*, and $\tau$ is the **temperature**.
3. This converts logits into a **probability distribution** — values between 0 and 1 that sum to 1.  
4. A token is then **sampled** from this distribution to produce the next word.


### Why this matters
The decoding step determines:
- **Creativity** of the output (via randomness)
- **Coherence** and **accuracy**
- **Likelihood of hallucinations**

If we always pick the most probable token, the model becomes deterministic and repetitive.  
If we sample too randomly, coherence and meaning degrade.

---

## Temperature

The **temperature** parameter $\tau$ controls how “sharp” or “flat” the probability distribution is.

- **Low temperature (≈ 0.1–0.5)** → Concentrated distribution  
  → The model behaves deterministically and repeats common phrases.

- **High temperature (≈ 1.0–2.0)** → Flattened distribution  
  → The model becomes more 'creative' or chaotic.

The visualization includes:
* **Probability**: Color represents probability of the chosen token (high: green, low: red)
* **Perplexity**: Color represents how mixed the token probability distribution is (white: probability concentrated on one token, red: high entropy)

Hover the tokens to see the probability distribution.

### 🧩 Exercise 1 — Exploring Temperature
This exercise shows how temperature affects text generation. Generate responses and observe the effects of temperature.

Try changing the temperature between 0 and 4.0 and observe how the responses differ. What do you notice regarding probability of the chosen tokens? What do you notice regarding the Perplexity?

In [21]:
messages = [
    {"role":"user", "content":"Please tell me something about the Helmholtz Zentrum Dresden-Rossendorf! Include the founding year!"},
]
response = client.chat.completions.create(
    messages=messages,
    model=model,
    max_tokens =200,
    logprobs=True, # send logprobs of tokens with response
    top_logprobs = 5, # sent top 5 alternative tokens with response
    temperature = 1.0 # <- TODO: Try out different values!
)

In [22]:
visualize(response, initial="Probability")

## Top-K and Top-P Sampling

Even after applying temperature, we still have thousands of possible tokens.  
To make generation more coherent, we can restrict the choice set using **Top-K** or **Top-P (nucleus)** sampling.

### Top-K Sampling
- Keep only the **K most probable tokens**.
- Renormalize their probabilities to sum to 1.
- Sample from this smaller set.

### Top-P (Nucleus) Sampling
- Instead of a fixed number *K*, keep the **smallest set of tokens whose cumulative probability ≥ P** (e.g., 0.9).
- Renormalize and sample.

### Visual Exploration

Head to [Transformer Explainer: LLM Transformer Model Visually Explained](https://poloclub.github.io/transformer-explainer/).

Click the **“Probability 🔎”** button to visualize:
- **Logits** (raw scores before softmax)
- **Scaled logits** (after temperature adjustment)
- **Probabilities** (after softmax)
- **Top-k / Top-p** selections (depending on sampling mode)

### 🧩 Exercise 2 — Interactive Exploration
Use the **Transformer Explainer** visualization:

1. **Top-K**: Modify *k* and observe how the distribution truncates.  
2. **Top-P**: Adjust *p* and notice how it changes the set of allowed tokens.  
3. Observe how these interact with **temperature** — low temperature + low K/P gives deterministic text, high values produce creative output.

### 🧩 Exercise 3 — Top-P for OpenAI Generation

This exercise shows how top-k and top-p affects text generation. The OpenAI API does not provide a `top_k` sampling mode.

Try out `top_p` sampling mode with different threshold probabilites and temperatures to see how it affects text generation.

In [23]:
messages = [
    {"role":"user", "content":"Please tell me something about the Helmholtz Zentrum Dresden-Rossendorf! Include the founding year!"},
]
response = client.chat.completions.create(
    messages=messages,
    model=model,
    max_tokens =500,
    logprobs=True,
    top_logprobs = 5, 
    temperature = 0.7, # <- modify
    top_p=0.5, # <- modify
)
visualize(response, "Perplexity")